In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from typing import TypedDict,Literal,Annotated
import os
from operator import add
from pydantic import BaseModel,Field
from langchain_core.prompts import PromptTemplate
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3,
    timeout=60,
    max_retries=3
)

In [ ]:
class ticketstate(TypedDict,total=False):
    customer_id:str
    ticket:str
    ticket_category:str
    urgency:str
    order_id:str
    order_details:str
    draft_reply:str
    review_score:int
    feedback:str
    attempts:int
    approved:bool
    final_reply:str
    history:Annotated[list[str],add]
    human_approved:bool
    rejection_reason:str

In [ ]:
class Ticketclassification(BaseModel):
    category:Literal["billing","technical","general"]=Field(description="categorise the customer complaint.")
    urgency:Literal["low","medium","high"] =Field(description="Find the urgency of the complaint")
structured_llm=llm.with_structured_output( Ticketclassification)


In [ ]:
class TicketReview(BaseModel):
    score:int=Field(description="give score 1 to 10")
    feedback:str=Field(description="Explain what is missing or needs improvement")
    approved:bool=Field(description="True if reply is good enough else false")
review_llm=llm.with_structured_output(TicketReview)

In [ ]:
classification_prompt=PromptTemplate(
    template="""
   You are a customer support ticket classifier.
  Analyze the following customer complaint:{ticket}

    Classify it into exactly one category:
    - billing
    - technical
    - general

    Also assign an urgency level:
    - low
    - medium
    - high

    Guidelines for classifying ticket:
    - Billing issues involving duplicate charges or unauthorized
      transactions can be high urgency.
    - App crashes or major technical failures can be high urgency.
    - General questions are usually low urgency.
    - Consider the actual impact on the customer.

    Return the result in the required structured format.
    """,
    input_variables=['ticket']
)

In [ ]:
review_prompt=PromptTemplate(
    template="""
   You are a strict customer support quality reviewer.

    Customer complaint:
    {ticket}

    Draft reply:
    {draft_reply}

    Check whether the draft:
    1. Addresses the customer's actual problem.
    2. Is polite and professional.
    3. Provides useful information.
    4. Does not make unsupported promises.

    Give a score between 1 and 10.
    Approve only if the reply is good enough.
    Provide clear feedback if improvement is needed.
""",
input_variables=["ticket","draft_reply"]

)


In [ ]:
def route_review(state: ticketstate):

    # If reviewer approves the draft
    if state.get("approved", False):

        if state["ticket_category"] == "billing":
            return "human_approval"

        return "send"

    # If maximum attempts reached
    if state.get("attempts", 0) >= 3:
        return "human_approval"

    # Retry the correct handler
    return "retry_" + state["ticket_category"]

In [ ]:
classification_chain = (
    classification_prompt
    | structured_llm
)
review_chain=review_prompt|review_llm

In [ ]:

import re

def classify_ticket(state: ticketstate):
    print("\n------------CLASSIFICATION NODE------")

    ticket = state["ticket"]

    result = classification_chain.invoke({
        "ticket": ticket
    })

    match = re.search(
        r"order\s*#?\s*(\d+)",
        ticket,
        re.IGNORECASE
    )

    # Extract order ID
    order_id = match.group(1) if match else ""

    print("Extracted Order ID:", order_id)

    return {
        "ticket_category": result.category,
        "urgency": result.urgency,
        "order_id": order_id
    }

In [ ]:
def review_ticket(state: ticketstate):

    result = review_chain.invoke({
        "ticket": state["ticket"],
        "draft_reply": state["draft_reply"]
    })

    print("Reviewer executed!")
    print("Review score:", result.score)
    print("Feedback:", result.feedback)

    return {
        "review_score": result.score,
        "feedback": result.feedback,
        "approved": result.approved
    }

In [ ]:
mock_orders = {
    "1042": {
        "status": "Delivered",
        "amount": 2000,
        "payment_status": "Paid",
        "duplicate_payment": True
    },

    "1043": {
        "status": "Processing",
        "amount": 5000,
        "payment_status": "Paid",
        "duplicate_payment": False
    },

    "1044": {
        "status": "Shipped",
        "amount": 4000,
        "payment_status": "Pending",
        "duplicate_payment": False
    },
    "1045": {
       "status": "Processing",
        "amount": 2500,
        "payment_status": "Paid",
        "duplicate_payment": False
        },
    
    "1046": {
            "status": "Shipped",
            "amount": 6000,
            "payment_status": "Pending",
            "duplicate_payment": False
        }
}
def lookup_order(order_id:str):
    order=mock_orders.get(order_id)
    if order is None:
        return "order id not found"
    return order


In [ ]:

def billing_handler(state: ticketstate):

    order_id = state.get("order_id", "")
    order_details = lookup_order(order_id)
    feedback = state.get("feedback", "")

    if isinstance(order_details, str):

        draft = f"""
Hello,

We could not find order {order_id}.
Please verify your order ID and contact us again.

Thank you.
"""

    else:
        status = order_details["status"]
        amount = order_details["amount"]
        payment_status = order_details["payment_status"]
        duplicate = order_details["duplicate_payment"]

        if duplicate:
            draft = f"""
Hello,

We checked your order {order_id}.

We found a duplicate payment of Rs. {amount}.
We sincerely apologize for the inconvenience.

Your concern has been forwarded to our billing team
for refund verification and processing.

Thank you for your patience.
"""
        else:
            draft = f"""
Hello,

We checked your order {order_id}.

Order status: {status}
Payment status: {payment_status}
Amount: Rs. {amount}

We could not find a duplicate payment in our records.
Please share additional transaction details so we can investigate.

Thank you.
"""

    if feedback:
        print("Previous reviewer feedback:", feedback)

    attempts = state.get("attempts", 0) + 1

    print("Billing handler Executed. Attempt:", attempts)

    return {
        "draft_reply": draft,
        "order_details": str(order_details),
        "attempts": attempts
    }


In [ ]:
def route_ticket(state:ticketstate):
    category=state["ticket_category"]
    if category == "billing":
        return "billing"

    elif category == "technical":
        return "technical"

    else:
        return "general"

In [ ]:

def human_approval(state: ticketstate):

    decision = interrupt({
        "message": "Please review the billing reply.",
        "ticket": state["ticket"],
        "draft_reply": state["draft_reply"]
    })

    if decision["approved"]:
        final_draft = decision.get("edited_reply") or state["draft_reply"]

        return {
            "human_approved": True,
            "final_reply": final_draft
        }

    return {
        "human_approved": False,
        "final_reply": "Your billing request has been escalated to our support team."
    }


In [ ]:
def route_human_decision(state: ticketstate):

    if state.get("human_approved"):
        return "send"

    return "end"

In [ ]:

def send_final_reply(state: ticketstate):

    final_reply = state.get("final_reply") or state["draft_reply"]

    print("\n========== FINAL CUSTOMER RESPONSE ==========")
    print(final_reply)

    return {
        "final_reply": final_reply
    }


In [ ]:

def rejection_handler(state: ticketstate):

    print("\nTicket rejected by human!")
    print("Sending ticket to manual support team.")

    rejection_message = (
        "Your request has been escalated to our manual support team."
    )

    return {
        "human_approved": False,
        "rejection_reason": "Human reviewer rejected the draft. Manual review required.",
        "final_reply": rejection_message
    }


In [ ]:
#checkpointer = InMemorySaver()
conn = sqlite3.connect(
    "support_tickets.db",
    check_same_thread=False
)

checkpointer = SqliteSaver(conn)

checkpointer.setup()
builder=StateGraph(ticketstate)
builder.add_node('classify',classify_ticket)
builder.add_node('billing',billing_handler)
builder.add_node('technical',technical_handler)
builder.add_node('general',general_handler)
builder.add_node('review',review_ticket)
builder.add_node('human_approval',human_approval)
builder.add_node('send',send_final_reply)
builder.add_node("rejection", rejection_handler)
                 
builder.add_edge(START,"classify")
builder.add_conditional_edges(
    "classify",route_ticket,
    {
        "billing": "billing",
        "technical": "technical",
        "general": "general"
    }
)
builder.add_edge("billing","review")
builder.add_edge("technical","review")
builder.add_edge("general","review")
builder.add_conditional_edges(
    "review",
    route_review,
     {
        "retry_billing": "billing",
        "retry_technical": "technical",
        "retry_general": "general",
        "human_approval": "human_approval",
        "send": "send"
    }
)
builder.add_conditional_edges(
    "human_approval",
    route_human_decision,
    {
        "send": "send",
        "end": "rejection"
    }
)
builder.add_edge("rejection", END)
builder.add_edge("send", END)
graph = builder.compile(checkpointer=checkpointer)


In [ ]:
from IPython.display import Image, display

graph_image = graph.get_graph().draw_mermaid_png()

with open("graph.png", "wb") as f:
    f.write(graph_image)

display(Image(graph_image))

In [ ]:
config = {
    "configurable": {
        "thread_id": "billing_test_1"
    }
}

result = graph.invoke(
    {
        "ticket": "I was charged twice for order #1042"
    },
    config=config
)

print(result)